In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F


# -------------------------------------
# CNN Block with GroupNorm and GELU
# -------------------------------------
class CNNBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        groups,
        downsample=True,
        dropout=0.0
    ):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.norm = nn.GroupNorm(
            groups, out_channels
        )

        self.pool = (
            nn.MaxPool2d(2)
            if downsample
            else nn.Identity()
        )

        self.dropout = (
            nn.Dropout2d(dropout)
            if dropout > 0
            else nn.Identity()
        )

    def forward(self, x):

        x = self.conv(x)
        x = self.norm(x)
        x = F.gelu(x)

        x = self.pool(x)
        x = self.dropout(x)

        return x


# -------------------------------------
# Identity-Preserving CNN Residual Module
# -------------------------------------
class ResidualRefinement(nn.Module):

    def __init__(self, channels, groups=16):
        super().__init__()

        bottleneck = channels // 4

        self.conv1 = nn.Conv2d(
            channels,
            bottleneck,
            kernel_size=1,
            bias=False
        )

        self.norm1 = nn.GroupNorm(
            8, bottleneck
        )

        self.conv2 = nn.Conv2d(
            bottleneck,
            bottleneck,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.norm2 = nn.GroupNorm(
            8, bottleneck
        )

        self.conv3 = nn.Conv2d(
            bottleneck,
            channels,
            kernel_size=1,
            bias=False
        )

        self.norm3 = nn.GroupNorm(
            groups, channels
        )

        # Initialize residual branch near zero
        nn.init.zeros_(self.norm3.weight)
        nn.init.zeros_(self.norm3.bias)

    def forward(self, x):

        identity = x

        out = self.conv1(x)
        out = F.gelu(self.norm1(out))

        out = self.conv2(out)
        out = F.gelu(self.norm2(out))

        out = self.conv3(out)
        out = self.norm3(out)

        # Identity-preserving residual mapping
        return identity + out


# -------------------------------------
# Custom CNN Backbone - Five Blocks
# -------------------------------------
class CustomCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Block 1: 3 -> 32
        self.block1 = CNNBlock(
            3, 32,
            groups=8
        )

        # Block 2: 32 -> 64
        self.block2 = CNNBlock(
            32, 64,
            groups=8
        )

        # Block 3: 64 -> 64
        self.block3 = CNNBlock(
            64, 64,
            groups=8,
            dropout=0.05
        )

        # Block 4: 64 -> 128
        self.block4 = CNNBlock(
            64, 128,
            groups=16,
            dropout=0.10
        )

        # Residual module after Block 4
        self.refine4 = ResidualRefinement(
            128, groups=16
        )

        # Block 5: 128 -> 256
        self.block5 = CNNBlock(
            128, 256,
            groups=16,
            downsample=False,
            dropout=0.15
        )

        # Residual module after Block 5
        self.refine5 = ResidualRefinement(
            256, groups=16
        )

        # Dedicated Grad-CAM convolution
        self.gradcam_conv = nn.Conv2d(
            256,
            256,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.gradcam_norm = nn.GroupNorm(
            16, 256
        )

        # Spatial pooling for graph construction
        self.graph_pool = nn.AdaptiveAvgPool2d(
            (7, 7)
        )

    def forward(self, x):

        # [B, 32, 112, 112]
        x = self.block1(x)

        # [B, 64, 56, 56]
        x = self.block2(x)

        # [B, 64, 28, 28]
        x = self.block3(x)

        # [B, 128, 14, 14]
        x = self.block4(x)
        x = self.refine4(x)

        # [B, 256, 14, 14]
        x = self.block5(x)
        x = self.refine5(x)

        # Dedicated Grad-CAM layer
        # [B, 256, 14, 14]
        x = self.gradcam_conv(x)

        x = self.gradcam_norm(x)
        x = F.gelu(x)

        # [B, 256, 7, 7]
        x = self.graph_pool(x)

        return x


# -------------------------------------
# Initial Spatial Adjacency Matrix
# -------------------------------------
def build_adjacency(height=7, width=7):

    n = height * width

    adjacency = torch.eye(n)

    for i in range(height):
        for j in range(width):

            idx = i * width + j

            for di, dj in [(1, 0), (0, 1)]:

                ni = i + di
                nj = j + dj

                if ni < height and nj < width:

                    neighbor = ni * width + nj

                    adjacency[idx, neighbor] = 1.0
                    adjacency[neighbor, idx] = 1.0

    return adjacency


# -------------------------------------
# Learnable Adjacency Matrix
# -------------------------------------
class LearnableAdjacency(nn.Module):

    def __init__(self, height=7, width=7):
        super().__init__()

        spatial_adjacency = build_adjacency(
            height, width
        )

        # Initialize strong local connections
        # and weak nonlocal connections
        initial_weights = torch.where(
            spatial_adjacency > 0,
            torch.full_like(
                spatial_adjacency, 0.95
            ),
            torch.full_like(
                spatial_adjacency, 0.005
            )
        )

        # Trainable adjacency parameters
        self.adjacency_logits = nn.Parameter(
            torch.logit(initial_weights)
        )

    def forward(self):

        # Symmetric adjacency
        logits = 0.5 * (
            self.adjacency_logits +
            self.adjacency_logits.T
        )

        # Positive learnable edge weights
        adjacency = torch.sigmoid(logits)

        # Degree normalization
        degree = adjacency.sum(
            dim=1
        ).clamp_min(1e-6)

        d_inv = degree.rsqrt()

        # Symmetric normalization
        adjacency = (
            d_inv.unsqueeze(1)
            * adjacency
            * d_inv.unsqueeze(0)
        )

        return adjacency


# -------------------------------------
# Residual Graph Convolution
# -------------------------------------
class ResidualGraphConvolution(nn.Module):

    def __init__(
        self,
        in_features,
        out_features,
        dropout=0.4
    ):
        super().__init__()

        self.linear = nn.Linear(
            in_features,
            out_features,
            bias=False
        )

        self.norm = nn.LayerNorm(
            out_features
        )

        self.dropout = nn.Dropout(dropout)

        # Residual projection when dimensions differ
        if in_features != out_features:

            self.skip = nn.Linear(
                in_features,
                out_features,
                bias=False
            )

        else:
            self.skip = nn.Identity()

    def forward(self, x, adjacency):

        identity = self.skip(x)

        # Linear feature transformation
        out = self.linear(x)

        # Graph feature aggregation
        out = torch.matmul(
            adjacency, out
        )

        # Normalization and activation
        out = self.norm(out)
        out = F.gelu(out)

        # Dropout
        out = self.dropout(out)

        # Residual connection
        out = identity + out

        return out


# -------------------------------------
# V4: Custom CNN + Learnable Residual GCN
# -------------------------------------
class Model(nn.Module):

    def __init__(
        self,
        num_classes,
        dropout=0.4
    ):
        super().__init__()

        # Five-block CNN backbone
        self.backbone = CustomCNN()

        # CNN node feature normalization
        self.feature_norm = nn.LayerNorm(256)

        # Learnable adjacency graph
        self.adjacency = LearnableAdjacency(
            height=7,
            width=7
        )

        # Three residual GCN layers
        self.gcn_layers = nn.ModuleList([

            # GCN 1: 256 -> 512
            ResidualGraphConvolution(
                256, 512, dropout
            ),

            # GCN 2: 512 -> 512
            ResidualGraphConvolution(
                512, 512, dropout
            ),

            # GCN 3: 512 -> 512
            ResidualGraphConvolution(
                512, 512, dropout
            )
        ])

        # Global mean pooling + LayerNorm
        self.global_norm = nn.LayerNorm(512)

        # Linear classification head
        self.classifier = nn.Linear(
            512, num_classes
        )

    def forward(self, x):

        # ---------------------------------
        # CNN Feature Extraction
        # ---------------------------------
        # [B, 256, 7, 7]
        x = self.backbone(x)

        # ---------------------------------
        # Spatial Features -> Graph Nodes
        # ---------------------------------
        # [B, 49, 256]
        x = x.flatten(2).transpose(1, 2)

        # Node feature normalization
        x = self.feature_norm(x)

        # ---------------------------------
        # Learnable Adjacency
        # ---------------------------------
        # [49, 49]
        adjacency = self.adjacency()

        # ---------------------------------
        # Three Residual GCN Layers
        # ---------------------------------
        for layer in self.gcn_layers:

            x = layer(x, adjacency)

        # [B, 49, 512]

        # ---------------------------------
        # Global Mean Pooling
        # ---------------------------------
        # [B, 512]
        x = x.mean(dim=1)

        # Layer normalization
        x = self.global_norm(x)

        # ---------------------------------
        # Classification
        # ---------------------------------
        logits = self.classifier(x)

        return logits


# -------------------------------------
# Model Initialization
# -------------------------------------
def create_model(num_classes):

    return Model(
        num_classes=num_classes,
        dropout=0.4
    )


# -------------------------------------
# Model Testing
# -------------------------------------
if __name__ == "__main__":

    torch.manual_seed(42)

    model = create_model(num_classes=5)

    inputs = torch.randn(
        2, 3, 224, 224
    )

    # Capture Grad-CAM feature shape
    feature_shapes = []

    hook = model.backbone.gradcam_conv.register_forward_hook(
        lambda module, inputs, output:
            feature_shapes.append(tuple(output.shape))
    )

    # Forward pass
    outputs = model(inputs)

    print("Output shape:", outputs.shape)

    print(
        "Grad-CAM feature map:",
        feature_shapes[-1]
    )

    print(
        "Adjacency shape:",
        model.adjacency().shape
    )

    # Model parameters
    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    print("Total parameters:", total_params)

    # Test backward propagation
    labels = torch.tensor([0, 1])

    loss = F.cross_entropy(
        outputs, labels
    )

    loss.backward()

    print(
        "Adjacency gradients:",
        model.adjacency.adjacency_logits.grad is not None
    )

    print(
        "Grad-CAM gradients:",
        model.backbone.gradcam_conv.weight.grad is not None
    )

    hook.remove()


In [ ]:
import random
import numpy as np


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = create_model(num_classes=len(class_names)).to(device)

model_name = "TomatoNet"